In [109]:
import pandas as pd
import os
import re
import numpy as np

In [110]:
SCRIPT_DIR_PATH = os.getcwd()
CB_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(CB_DIR_PATH)
TORNADO_DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
INPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "input")
OUTPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "output")

In [111]:
def add_sector_and_transformation_fields(df: pd.DataFrame, strategy_col: str = "strategy") -> pd.DataFrame:
    """
    Creates:
      - sector: sector code (e.g., AGRC)
      - transformation_name: transformation text after '... - <SECTOR>:'
    """
    df = df.copy()

    # --- sector extraction (captures 3-6 uppercase letters before colon) ---
    # Example: "Singleton - Default Value - AGRC: Improve rice..." -> AGRC
    df["sector"] = df[strategy_col].str.extract(r"-\s*([A-Z]{3,6})\s*:", expand=False)

    # Special case: baseline strategy
    df.loc[df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False), "sector"] = "BASE"

    # --- transformation_name extraction ---
    # Keep only text after "<SECTOR>:"
    # Example -> "Improve rice..."
    df["transformation_name"] = df[strategy_col].str.extract(r":\s*(.*)$", expand=False)

    # If it's baseline, keep the full strategy string as the name (or label it as BASE)
    base_mask = df[strategy_col].str.contains(r"^Strategy\s+TX:BASE", regex=True, na=False)
    df.loc[base_mask, "transformation_name"] = "BASE"

    # Clean whitespace
    df["transformation_name"] = df["transformation_name"].fillna("").str.strip()

    return df

## Load and process emission data

In [112]:
# Load the decomposed emissions long format data
emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "decomposed_emissions_bulgaria_2022_tornado.csv"))
# emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "decomposed_emissions_bulgaria_2022_trww_debug.csv"))
emissions_df.head()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
0,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125150,2022,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125150,0.125150
1,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125247,2023,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125247,0.125247
2,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125338,2024,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125338,0.125338
3,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125418,2025,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125418,0.125418
4,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125484,2026,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125484,0.125484


In [113]:
print(emissions_df.primary_id.nunique())

49


In [114]:
# check unique strategy
emissions_df['strategy'].unique()

array(['Strategy TX:BASE', 'WEM',
       'Singleton - WAM Value - PFLO: Industrial carbon capture and sequestration',
       'Singleton - WAM Value - TRNS: Mode shift passenger vehicles to others',
       'Singleton - WAM Value - ENTC: 95% of electricity is generated by renewables in final time period',
       'Singleton - WAM Value - WASO: Increase landfilling',
       'Singleton - WAM Value - WASO: Increase recycling',
       'Singleton - WAM Value - LSMM: Improve manure management for poultry',
       'Singleton - WAM Value - AGRC: Improve rice management',
       'Singleton - WAM Value - TRNS: Electrify rail',
       'Singleton - WAM Value - ENTC: Reduce transmission losses',
       'Singleton - WAM Value - WASO: Consumer food waste reduction',
       'Singleton - WAM Value - ENTC: Clean hydrogen',
       'Singleton - WAM Value - FGTV: Minimize leaks',
       'Singleton - WAM Value - SCOE: Reduce end-use demand for heat energy by improving building shell',
       'Singleton - WAM V

In [115]:
# Drop historical and tx:base from df
filtered_emissions_df = emissions_df.loc[~emissions_df['strategy'].isin(['Historical', 'Strategy TX:BASE'])]
print(emissions_df['strategy'].nunique())
print(filtered_emissions_df['strategy'].nunique())

50
48


In [116]:
filtered_emissions_df["strategy"].unique()

array(['WEM',
       'Singleton - WAM Value - PFLO: Industrial carbon capture and sequestration',
       'Singleton - WAM Value - TRNS: Mode shift passenger vehicles to others',
       'Singleton - WAM Value - ENTC: 95% of electricity is generated by renewables in final time period',
       'Singleton - WAM Value - WASO: Increase landfilling',
       'Singleton - WAM Value - WASO: Increase recycling',
       'Singleton - WAM Value - LSMM: Improve manure management for poultry',
       'Singleton - WAM Value - AGRC: Improve rice management',
       'Singleton - WAM Value - TRNS: Electrify rail',
       'Singleton - WAM Value - ENTC: Reduce transmission losses',
       'Singleton - WAM Value - WASO: Consumer food waste reduction',
       'Singleton - WAM Value - ENTC: Clean hydrogen',
       'Singleton - WAM Value - FGTV: Minimize leaks',
       'Singleton - WAM Value - SCOE: Reduce end-use demand for heat energy by improving building shell',
       'Singleton - WAM Value - TRNS: SHIFT_M

In [117]:
filtered_emissions_df.tail()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
66782,6067.0,126126.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.001754,2046,N2O,0.0,0.0,Singleton - WAM Value - WALI: Improved urban w...,BGR,bulgaria,SISEPUEDE,0.001754,NaN
66783,6067.0,126126.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.001815,2047,N2O,0.0,0.0,Singleton - WAM Value - WALI: Improved urban w...,BGR,bulgaria,SISEPUEDE,0.001815,NaN
66784,6067.0,126126.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.001876,2048,N2O,0.0,0.0,Singleton - WAM Value - WALI: Improved urban w...,BGR,bulgaria,SISEPUEDE,0.001876,NaN
66785,6067.0,126126.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.001935,2049,N2O,0.0,0.0,Singleton - WAM Value - WALI: Improved urban w...,BGR,bulgaria,SISEPUEDE,0.001935,NaN
66786,6067.0,126126.0,Waste - Wastewater Treatment:N2O,Waste,Waste - Wastewater Treatment,0.001994,2050,N2O,0.0,0.0,Singleton - WAM Value - WALI: Improved urban w...,BGR,bulgaria,SISEPUEDE,0.001994,NaN


In [118]:
# Load decomposed data from original run containing base, wem and wam
original_decomposed_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "decomposed_emissions_bulgaria_2022_original.csv"))


original_decomposed_df.strategy.unique()

array(['Strategy TX:BASE', 'WEM', 'WAM', 'WAM_F', 'WAM_F_NO_CCS',
       'Historical'], dtype=object)

In [119]:
# Keep only base strategy in the original df
original_base_df = original_decomposed_df.loc[original_decomposed_df['strategy'] == 'Strategy TX:BASE']
original_base_df.strategy.unique()

array(['Strategy TX:BASE'], dtype=object)

In [120]:
original_base_df.head()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
0,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125150,2022,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125150,0.125150
1,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125247,2023,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125247,0.125247
2,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125338,2024,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125338,0.125338
3,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125418,2025,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125418,0.125418
4,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125484,2026,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125484,0.125484


In [121]:
# Now concat the original base df and the filtered emissions df
tornado_emissions_df = pd.concat([original_base_df, filtered_emissions_df], ignore_index=True)
tornado_emissions_df['strategy'].unique()

array(['Strategy TX:BASE', 'WEM',
       'Singleton - WAM Value - PFLO: Industrial carbon capture and sequestration',
       'Singleton - WAM Value - TRNS: Mode shift passenger vehicles to others',
       'Singleton - WAM Value - ENTC: 95% of electricity is generated by renewables in final time period',
       'Singleton - WAM Value - WASO: Increase landfilling',
       'Singleton - WAM Value - WASO: Increase recycling',
       'Singleton - WAM Value - LSMM: Improve manure management for poultry',
       'Singleton - WAM Value - AGRC: Improve rice management',
       'Singleton - WAM Value - TRNS: Electrify rail',
       'Singleton - WAM Value - ENTC: Reduce transmission losses',
       'Singleton - WAM Value - WASO: Consumer food waste reduction',
       'Singleton - WAM Value - ENTC: Clean hydrogen',
       'Singleton - WAM Value - FGTV: Minimize leaks',
       'Singleton - WAM Value - SCOE: Reduce end-use demand for heat energy by improving building shell',
       'Singleton - WAM V

In [122]:
tornado_emissions_df.head()

,strategy_id,primary_id,Edgar_Class,CSC.Sector,CSC.Subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source,value_original,value_hp
0,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125150,2022,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125150,0.125150
1,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125247,2023,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125247,0.125247
2,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125338,2024,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125338,0.125338
3,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125418,2025,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125418,0.125418
4,0.0,0.0,AG - Crops:CH4,Agriculture,AG - Crops,0.125484,2026,CH4,0.0,0.0,Strategy TX:BASE,BGR,bulgaria,SISEPUEDE,0.125484,0.125484


In [123]:
# Keep only relevant CSC.Subsectors

relevant_subsectors = [
    "AG - Crops",
    "AG - Livestock",
    "IN - Industrial Processes",
    "LULUCF - Forest Land",
    "LULUCF - HWP",
    "LULUCF - Wetlands",
    "LULUCF - Cropland",
    "LULUCF - Grassland",
    "LULUCF - Settlements",
    "LULUCF - Other Land",
    "Waste - Solid Waste",
    "Waste - Wastewater Treatment"
]
print(tornado_emissions_df.shape)
tornado_emissions_df = tornado_emissions_df.loc[tornado_emissions_df['CSC.Subsector'].isin(relevant_subsectors)]
print(tornado_emissions_df.shape)

(66787, 16)
(36946, 16)


In [124]:
# Aggregate by strategy_id, primary_id and strategy, and sum value
tornado_emissions_agg_df = tornado_emissions_df.groupby(
    ['strategy_id', 'primary_id', 'strategy']
)['value'].sum().reset_index()

tornado_emissions_agg_df.head()


,strategy_id,primary_id,strategy,value
0,0.0,0.0,Strategy TX:BASE,130.422973
1,6003.0,74074.0,WEM,102.531321
2,6006.0,77077.0,Singleton - WAM Value - PFLO: Industrial carbo...,112.769234
3,6008.0,78078.0,Singleton - WAM Value - TRNS: Mode shift passe...,130.422973
4,6009.0,79079.0,Singleton - WAM Value - ENTC: 95% of electrici...,130.422973


In [125]:
tornado_emissions_agg_df.tail()

,strategy_id,primary_id,strategy,value
44,6054.0,119119.0,Singleton - WAM Value - PFLO: WASO actions to...,125.473507
45,6055.0,120120.0,Singleton - WAM Value - PFLO: WALI actions to...,129.788385
46,6059.0,123123.0,Singleton - WAM Value - PFLO: LSMM actions to ...,123.947918
47,6066.0,125125.0,Singleton - WAM Value - WALI: Improved rural w...,129.903183
48,6067.0,126126.0,Singleton - WAM Value - WALI: Improved urban w...,134.402240


In [126]:
# check if strategy id nunique matches amount of rows
print(tornado_emissions_agg_df['strategy_id'].nunique())
print(tornado_emissions_agg_df.shape[0])

49
49


In [127]:
# rename value to emission_total
tornado_emissions_agg_df = tornado_emissions_agg_df.rename(columns={'value': 'emission_total'})

# create base_emission_total column by setting it to the strategy_id == 0 value
base_emission_total = tornado_emissions_agg_df.loc[tornado_emissions_agg_df['strategy_id'] == 0, 'emission_total'].values[0]
tornado_emissions_agg_df['base_emission_total'] = base_emission_total

# calculate emission difference column
tornado_emissions_agg_df['emission_diff'] =  tornado_emissions_agg_df['emission_total'] - tornado_emissions_agg_df['base_emission_total']
tornado_emissions_agg_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff
0,0.0,0.0,Strategy TX:BASE,130.422973,130.422973,0.000000e+00
1,6003.0,74074.0,WEM,102.531321,130.422973,-2.789165e+01
2,6006.0,77077.0,Singleton - WAM Value - PFLO: Industrial carbo...,112.769234,130.422973,-1.765374e+01
3,6008.0,78078.0,Singleton - WAM Value - TRNS: Mode shift passe...,130.422973,130.422973,-5.375315e-08
4,6009.0,79079.0,Singleton - WAM Value - ENTC: 95% of electrici...,130.422973,130.422973,-5.375315e-08


In [128]:
tornado_emissions_agg_extended_df = add_sector_and_transformation_fields(tornado_emissions_agg_df)
tornado_emissions_agg_extended_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
0,0.0,0.0,Strategy TX:BASE,130.422973,130.422973,0.000000e+00,BASE,BASE
1,6003.0,74074.0,WEM,102.531321,130.422973,-2.789165e+01,NaN,
2,6006.0,77077.0,Singleton - WAM Value - PFLO: Industrial carbo...,112.769234,130.422973,-1.765374e+01,PFLO,Industrial carbon capture and sequestration
3,6008.0,78078.0,Singleton - WAM Value - TRNS: Mode shift passe...,130.422973,130.422973,-5.375315e-08,TRNS,Mode shift passenger vehicles to others
4,6009.0,79079.0,Singleton - WAM Value - ENTC: 95% of electrici...,130.422973,130.422973,-5.375315e-08,ENTC,95% of electricity is generated by renewables ...


In [129]:
tornado_emissions_agg_extended_df.to_clipboard(index=False)

## Load and process CB data

In [130]:
# cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-01-29T15;28;40.322709_tornado_raw.csv"))
cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-02-09T12;59;59.346494_tornado_raw.csv"))
cb_raw_df.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value
0,PFLO:WEM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
1,PFLO:WEM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
2,PFLO:WEM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145804,752911.145804,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0
3,PFLO:WEM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257771,746564.257771,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0
4,PFLO:WEM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204939,740262.204939,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0


In [131]:
# --- Create a copy of the raw data ---
cb_data = cb_raw_df.copy()

# Split 'variable' into components: name, sector, cb_type, item_1, item_2
# (Assumes exactly 5 colon-separated parts; if there are more colons inside the last field,
# they will be kept in item_2 thanks to n=4)
cb_chars = cb_data["variable"].astype(str).str.split(":", n=4, expand=True)
cb_chars.columns = ["name", "sector", "cb_type", "item_1", "item_2"]
cb_data = pd.concat([cb_data, cb_chars], axis=1)
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2
0,PFLO:WEM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
1,PFLO:WEM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
2,PFLO:WEM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145804,752911.145804,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural
3,PFLO:WEM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257771,746564.257771,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural
4,PFLO:WEM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204939,740262.204939,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural


In [132]:
# Scale value from USD to billions (divide by 1e9)
if "value" in cb_data.columns:
    cb_data["value"] = cb_data["value"] / 1e9

# --- Remove "shifted" entries ---
# # Remove rows where item_2 contains "shifted"
# cb_data = cb_data[~cb_data["item_2"].astype(str).str.contains("shifted", na=False)]

# # Remove any remaining rows where variable contains "shifted2"
# cb_data = cb_data[~cb_data["variable"].astype(str).str.contains("shifted2", na=False)]

# --- Add Year column (Year = time_period + 2015) ---
cb_data["Year"] = cb_data["time_period"] + 2015

cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year
0,PFLO:WEM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0
1,PFLO:WEM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0
2,PFLO:WEM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145804,752911.145804,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0
3,PFLO:WEM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257771,746564.257771,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0
4,PFLO:WEM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204939,740262.204939,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0


In [133]:
# Load attribute strategy
attribute_strategy_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "ATTRIBUTE_STRATEGY.csv"))
attribute_strategy_df = attribute_strategy_df[["strategy_id", "strategy_code"]]
attribute_strategy_df.head()

,strategy_id,strategy_code
0,0,BASE
1,1000,AGRC:DEC_CH4_RICE
2,1001,AGRC:DEC_EXPORTS
3,1002,AGRC:DEC_LOSSES_SUPPLY_CHAIN
4,1003,AGRC:INC_CONSERVATION_AGRICULTURE


In [134]:
attribute_strategy_df.strategy_id.unique()

array([   0, 1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009,
       1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020,
       1021, 1022, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008,
       2009, 2010, 2011, 2012, 3000, 3001, 3002, 3003, 3004, 3005, 3006,
       3007, 3008, 3009, 3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017,
       3018, 3019, 3020, 3021, 3022, 3023, 3024, 3025, 3026, 4000, 4001,
       4002, 4003, 4004, 4005, 4006, 6000, 6001, 6002, 6003, 6004, 6005,
       6006, 6008, 6009, 6010, 6012, 6013, 6014, 6015, 6016, 6017, 6018,
       6019, 6020, 6021, 6022, 6023, 6024, 6025, 6027, 6028, 6030, 6031,
       6032, 6033, 6034, 6035, 6037, 6038, 6039, 6040, 6041, 6043, 6044,
       6045, 6046, 6047, 6048, 6049, 6050, 6051, 6052, 6053, 6054, 6055,
       6066, 6067, 6057, 6058, 6059, 6060])

In [135]:
attribute_strategy_df.strategy_code.unique()

array(['BASE', 'AGRC:DEC_CH4_RICE', 'AGRC:DEC_EXPORTS',
       'AGRC:DEC_LOSSES_SUPPLY_CHAIN',
       'AGRC:INC_CONSERVATION_AGRICULTURE', 'AGRC:INC_PRODUCTIVITY',
       'FRST:INCREASE_SEQUESTRATION', 'LNDU:BOUND_CLASSES',
       'LNDU:DEC_CLASS_LOSS', 'LNDU:DEC_DEFORESTATION',
       'LNDU:DEC_SOC_LOSS_PASTURES', 'LNDU:INC_REFORESTATION',
       'LNDU:INC_SILVOPASTURE', 'LNDU:PLUR', 'LSMM:INC_CAPTURE_BIOGAS',
       'LSMM:INC_MANAGEMENT_CATTLE_PIGS', 'LSMM:INC_MANAGEMENT_OTHER',
       'LSMM:INC_MANAGEMENT_POULTRY', 'LVST:DEC_ENTERIC_FERMENTATION',
       'LVST:DEC_EXPORTS', 'LVST:INC_PRODUCTIVITY',
       'SOIL:DEC_LIME_APPLIED', 'SOIL:DEC_N_APPLIED', 'AF:ALL',
       'TRWW:INC_CAPTURE_BIOGAS', 'TRWW:INC_COMPLIANCE_SEPTIC',
       'WALI:INC_TREATMENT_INDUSTRIAL', 'WALI:INC_TREATMENT_RURAL',
       'WALI:INC_TREATMENT_URBAN', 'WASO:DEC_CONSUMER_FOOD_WASTE',
       'WASO:INC_ANAEROBIC_AND_COMPOST', 'WASO:INC_CAPTURE_BIOGAS',
       'WASO:INC_ENERGY_FROM_BIOGAS', 'WASO:INC_ENERGY_FROM_

In [136]:
# Merge with cb_data on strategy_code
cb_data = cb_data.merge(attribute_strategy_df, on="strategy_code", how="left")
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:WEM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0,6003
1,PFLO:WEM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0,6003
2,PFLO:WEM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145804,752911.145804,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0,6003
3,PFLO:WEM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257771,746564.257771,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0,6003
4,PFLO:WEM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204939,740262.204939,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0,6003


In [137]:
cb_data[cb_data.strategy_code == "PFLO:WALI_BIOGAS_ACTIONS_WAM"].to_clipboard(index=False)

In [138]:
cb_data.strategy_code.unique()

array(['PFLO:WEM', 'PFLO:INC_IND_CCS_WAM',
       'TRNS:SHIFT_MODE_PASSENGER_WAM', 'ENTC:TARGET_RENEWABLE_ELEC_WAM',
       'WASO:INC_LANDFILLING_WAM', 'WASO:INC_RECYCLING_WAM',
       'LSMM:INC_MANAGEMENT_POULTRY_WAM', 'AGRC:DEC_CH4_RICE_WAM',
       'TRNS:SHIFT_FUEL_RAIL_WAM', 'ENTC:DEC_LOSSES_WAM',
       'WASO:DEC_CONSUMER_FOOD_WASTE_WAM',
       'ENTC:TARGET_CLEAN_HYDROGEN_WAM', 'FGTV:DEC_LEAKS_WAM',
       'SCOE:DEC_DEMAND_HEAT_WAM', 'TRNS:SHIFT_MODE_FREIGHT_WAM',
       'LSMM:INC_CAPTURE_BIOGAS_WAM',
       'WASO:INC_ENERGY_FROM_INCINERATION_WAM', 'CCSQ:INC_CAPTURE_WAM',
       'IPPU:DEC_HFCS_WAM', 'LSMM:INC_MANAGEMENT_CATTLE_PIGS_WAM',
       'TRNS:SHIFT_MODE_REGIONAL_WAM',
       'TRNS:INC_OCCUPANCY_LIGHT_DUTY_WAM',
       'INEN:INC_EFFICIENCY_ENERGY_WAM',
       'WASO:INC_ANAEROBIC_AND_COMPOST_WAM',
       'LSMM:INC_MANAGEMENT_OTHER_WAM', 'INEN:SHIFT_FUEL_HEAT_WAM',
       'INEN:INC_EFFICIENCY_PRODUCTION_WAM', 'SOIL:DEC_N_APPLIED_WAM',
       'FGTV:INC_FLARE_WAM', 'SCOE:INC_E

In [139]:
cb_data.strategy_id.unique()

array([6003, 6006, 6008, 6009, 6010, 6012, 6013, 6014, 6015, 6016, 6017,
       6018, 6019, 6020, 6021, 6022, 6023, 6024, 6025, 6027, 6028, 6030,
       6031, 6032, 6033, 6034, 6035, 6037, 6038, 6039, 6040, 6041, 6043,
       6044, 6045, 6046, 6047, 6048, 6049, 6050, 6051, 6052, 6053, 6054,
       6055, 6059, 6066, 6067])

In [140]:
# check for nans in strategy_id
cb_data[cb_data['strategy_id'].isna()]['strategy_code'].unique()

array([], dtype=object)

In [141]:
cb_data["sector"].unique()

array(['wali', 'entc', 'trns', 'lndu', 'waso', 'trww', 'lvst', 'agrc',
       'ccsq', 'inen', 'scoe', 'ippu', 'soil', 'lsmm', 'fgtv'],
      dtype=object)

In [142]:
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:WEM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0,6003
1,PFLO:WEM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0,6003
2,PFLO:WEM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145804,752911.145804,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0,6003
3,PFLO:WEM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257771,746564.257771,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0,6003
4,PFLO:WEM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204939,740262.204939,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0,6003


In [143]:
# filter sectors
target_sectors = ["wali", "trww", "waso", "soil", "ippu", "lvst", "agrc", "lndu", "lsmm"]
cb_data = cb_data[cb_data["sector"].isin(target_sectors)].copy()
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:WEM,0.0,bulgaria,7.0,pop_unimproved_rural,772936.249506,772936.249506,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2022.0,6003
1,PFLO:WEM,0.0,bulgaria,8.0,pop_unimproved_rural,758203.739880,758203.739880,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2023.0,6003
2,PFLO:WEM,0.0,bulgaria,9.0,pop_unimproved_rural,752911.145804,752911.145804,0.0,cb:wali:technical_cost:sanitation:unimp_rural,0.0,cb,wali,technical_cost,sanitation,unimp_rural,2024.0,6003
3,PFLO:WEM,0.0,bulgaria,10.0,pop_unimproved_rural,746564.257771,746564.257771,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2025.0,6003
4,PFLO:WEM,0.0,bulgaria,11.0,pop_unimproved_rural,740262.204939,740262.204939,0.0,cb:wali:technical_cost:sanitation:unimp_rural,-0.0,cb,wali,technical_cost,sanitation,unimp_rural,2026.0,6003


In [144]:
# aggregate sum(value) grouped by strategy_id and cb_type
cb_data = (
    cb_data.groupby(["strategy_id", "cb_type"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "cumulative"})
)
cb_data.head()

,strategy_id,cb_type,cumulative
0,6003,air_pollution,0.000000
1,6003,crop_value,-0.095978
2,6003,ecosystem_services,1.213763
3,6003,env_pollution,3.453008
4,6003,fuel_cost,2.387682


In [145]:
# unique cb_data types
cb_cats = cb_data["cb_type"].unique().tolist()

# long -> wide (R dcast equivalent)
wide_cb = (
    cb_data.pivot(index="strategy_id", columns="cb_type", values="cumulative")
      .reset_index()
)

# optional: remove column name from pivot for nicer printing
wide_cb.columns.name = None
wide_cb.head()

,strategy_id,air_pollution,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,land_pollution,lvst_value,technical_cost,water_pollution
0,6003,0.0,NaN,-0.095978,1.213763,3.453008,2.387682,11.198522,0.103508,0.001819,-0.000091,-3.023209,1.216747
1,6006,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.076456,0.000000
2,6008,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,6009,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,6010,0.0,NaN,0.000000,0.000000,3.038625,0.000000,0.000000,0.000000,0.000000,0.000000,-0.016031,0.000000


In [146]:
cb_cats

['air_pollution',
 'crop_value',
 'ecosystem_services',
 'env_pollution',
 'fuel_cost',
 'human_health',
 'ippu_value',
 'land_pollution',
 'lvst_value',
 'technical_cost',
 'water_pollution',
 'consumer_savings']

In [147]:
# --- 1) net_benefit = rowSums over all cb categories ---
wide_cb["net_benefit"] = wide_cb[cb_cats].sum(axis=1, skipna=True)

# --- 2) additional_benefits = rowSums excluding "technical_cost" ---
benefit_cols = [c for c in cb_cats if c != "technical_cost"]
wide_cb["additional_benefits"] = wide_cb[benefit_cols].sum(axis=1, skipna=True)

# --- 3) total_transformation_costs = rowSums over specific cols ---
cost_cols = ["technical_cost", "technical_savings", "fuel_cost"]

# (safe version: only use cols that exist in the df)
cost_cols = [c for c in cost_cols if c in wide_cb.columns]

wide_cb["total_transformation_costs"] = wide_cb[cost_cols].sum(axis=1, skipna=True)
wide_cb.head()

,strategy_id,air_pollution,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,land_pollution,lvst_value,technical_cost,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6003,0.0,NaN,-0.095978,1.213763,3.453008,2.387682,11.198522,0.103508,0.001819,-0.000091,-3.023209,1.216747,16.455771,19.478979,-0.635527
1,6006,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.076456,0.000000,-3.076456,0.000000,-3.076456
2,6008,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,6009,0.0,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,6010,0.0,NaN,0.000000,0.000000,3.038625,0.000000,0.000000,0.000000,0.000000,0.000000,-0.016031,0.000000,3.022595,3.038625,-0.016031


## Merge emissions and cb data and save

In [148]:
tornado_emissions_agg_extended_df[tornado_emissions_agg_extended_df.strategy == "Singleton - WAM Value - PFLO: LSMM actions to increase biogas capture"]

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
46,6059.0,123123.0,Singleton - WAM Value - PFLO: LSMM actions to ...,123.947918,130.422973,-6.475054,PFLO,LSMM actions to increase biogas capture


In [149]:
wide_cb.strategy_id.unique()

array([6003, 6006, 6008, 6009, 6010, 6012, 6013, 6014, 6015, 6016, 6017,
       6018, 6019, 6020, 6021, 6022, 6023, 6024, 6025, 6027, 6028, 6030,
       6031, 6032, 6033, 6034, 6035, 6037, 6038, 6039, 6040, 6041, 6043,
       6044, 6045, 6046, 6047, 6048, 6049, 6050, 6051, 6052, 6053, 6054,
       6055, 6059, 6066, 6067])

In [150]:
print(wide_cb.shape)
print(tornado_emissions_agg_extended_df.shape)

(48, 16)
(49, 8)


In [151]:
df_merged = pd.merge(
    tornado_emissions_agg_extended_df,
    wide_cb,
    on="strategy_id",
    how="inner"
)

df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,consumer_savings,...,fuel_cost,human_health,ippu_value,land_pollution,lvst_value,technical_cost,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6003.0,74074.0,WEM,102.531321,130.422973,-2.789165e+01,NaN,,0.0,NaN,...,2.387682,11.198522,0.103508,0.001819,-0.000091,-3.023209,1.216747,16.455771,19.478979,-0.635527
1,6006.0,77077.0,Singleton - WAM Value - PFLO: Industrial carbo...,112.769234,130.422973,-1.765374e+01,PFLO,Industrial carbon capture and sequestration,0.0,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,-3.076456,0.000000,-3.076456,0.000000,-3.076456
2,6008.0,78078.0,Singleton - WAM Value - TRNS: Mode shift passe...,130.422973,130.422973,-5.375315e-08,TRNS,Mode shift passenger vehicles to others,0.0,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,6009.0,79079.0,Singleton - WAM Value - ENTC: 95% of electrici...,130.422973,130.422973,-5.375315e-08,ENTC,95% of electricity is generated by renewables ...,0.0,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,6010.0,80080.0,Singleton - WAM Value - WASO: Increase landfil...,129.530159,130.422973,-8.928139e-01,WASO,Increase landfilling,0.0,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,-0.016031,0.000000,3.022595,3.038625,-0.016031


In [152]:
print(df_merged.shape)

(48, 23)


### Below we have some hardcoded fixed exclusive of this study case to replace incorrect tranformation names

In [153]:
# Update tranformation name for strategy id 6049
df_merged.loc[df_merged['strategy_id'] == 6049, 'transformation_name'] = "Increase solid waste biogas capture"
df_merged.loc[df_merged['strategy_id'] == 6021, 'transformation_name'] = "Shift mode for freight transport"
df_merged.loc[df_merged['strategy_id'] == 6028, 'transformation_name'] = "Shift mode for regional transport"
df_merged.loc[df_merged['strategy_id'] == 6038, 'transformation_name'] = "Increase flaring"

In [154]:
df_merged.columns

Index(['strategy_id', 'primary_id', 'strategy', 'emission_total',
       'base_emission_total', 'emission_diff', 'sector', 'transformation_name',
       'air_pollution', 'consumer_savings', 'crop_value', 'ecosystem_services',
       'env_pollution', 'fuel_cost', 'human_health', 'ippu_value',
       'land_pollution', 'lvst_value', 'technical_cost', 'water_pollution',
       'net_benefit', 'additional_benefits', 'total_transformation_costs'],
      dtype='object')

In [155]:
# multiply technical_cost by -1 to get positive costs
df_merged['technical_cost'] = df_merged['technical_cost'] * -1

# create marginal total abatement cost column
df_merged['marginal_total_abatement_cost_(USD/tCO2e)'] = (df_merged['technical_cost'] / df_merged['emission_diff'])*1000

# If technical_cost is positive then marginal_total_abatement_cost should be positive too.
# df_merged["marginal_total_abatement_cost_(USD/tCO2e)"] = np.where(df_merged["technical_cost"] > 0, df_merged["marginal_total_abatement_cost_(USD/tCO2e)"].abs(), df_merged["marginal_total_abatement_cost_(USD/tCO2e)"])
df_merged["marginal_total_abatement_cost_(USD/tCO2e)"] = df_merged["marginal_total_abatement_cost_(USD/tCO2e)"].abs() * np.sign(df_merged["technical_cost"])


In [156]:
df_merged["strategy"].unique()

array(['WEM',
       'Singleton - WAM Value - PFLO: Industrial carbon capture and sequestration',
       'Singleton - WAM Value - TRNS: Mode shift passenger vehicles to others',
       'Singleton - WAM Value - ENTC: 95% of electricity is generated by renewables in final time period',
       'Singleton - WAM Value - WASO: Increase landfilling',
       'Singleton - WAM Value - WASO: Increase recycling',
       'Singleton - WAM Value - LSMM: Improve manure management for poultry',
       'Singleton - WAM Value - AGRC: Improve rice management',
       'Singleton - WAM Value - TRNS: Electrify rail',
       'Singleton - WAM Value - ENTC: Reduce transmission losses',
       'Singleton - WAM Value - WASO: Consumer food waste reduction',
       'Singleton - WAM Value - ENTC: Clean hydrogen',
       'Singleton - WAM Value - FGTV: Minimize leaks',
       'Singleton - WAM Value - SCOE: Reduce end-use demand for heat energy by improving building shell',
       'Singleton - WAM Value - TRNS: SHIFT_M

In [157]:
df_merged.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot.csv"), index=False)

### Create a QA version

In [158]:
df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,consumer_savings,...,human_health,ippu_value,land_pollution,lvst_value,technical_cost,water_pollution,net_benefit,additional_benefits,total_transformation_costs,marginal_total_abatement_cost_(USD/tCO2e)
0,6003.0,74074.0,WEM,102.531321,130.422973,-2.789165e+01,NaN,,0.0,NaN,...,11.198522,0.103508,0.001819,-0.000091,3.023209,1.216747,16.455771,19.478979,-0.635527,108.391177
1,6006.0,77077.0,Singleton - WAM Value - PFLO: Industrial carbo...,112.769234,130.422973,-1.765374e+01,PFLO,Industrial carbon capture and sequestration,0.0,NaN,...,0.000000,0.000000,0.000000,0.000000,3.076456,0.000000,-3.076456,0.000000,-3.076456,174.266545
2,6008.0,78078.0,Singleton - WAM Value - TRNS: Mode shift passe...,130.422973,130.422973,-5.375315e-08,TRNS,Mode shift passenger vehicles to others,0.0,NaN,...,0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,6009.0,79079.0,Singleton - WAM Value - ENTC: 95% of electrici...,130.422973,130.422973,-5.375315e-08,ENTC,95% of electricity is generated by renewables ...,0.0,NaN,...,0.000000,0.000000,0.000000,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,6010.0,80080.0,Singleton - WAM Value - WASO: Increase landfil...,129.530159,130.422973,-8.928139e-01,WASO,Increase landfilling,0.0,NaN,...,0.000000,0.000000,0.000000,0.000000,0.016031,0.000000,3.022595,3.038625,-0.016031,17.955288


In [159]:
df_merged.sector.unique()

array([nan, 'PFLO', 'TRNS', 'ENTC', 'WASO', 'LSMM', 'AGRC', 'FGTV',
       'SCOE', 'CCSQ', 'IPPU', 'INEN', 'SOIL', 'TRDE', 'WALI', 'LVST',
       'LNDU'], dtype=object)

In [160]:
relevant_sectors = [
    "AGRC",
    "LVST",
    "IPPU",
    "SOIL",
    "WALI",
    "TRWW",
    "WASO",
    "LNDU",
    "LSMM",
    "PFLO"
]

# keep only relevant sectors
df_merged_filtered = df_merged.loc[df_merged['sector'].isin(relevant_sectors)]

relevant_fields = [
    "transformation_name",
    "sector",
    "base_emission_total",
    "emission_total",
    "emission_diff",
    "technical_cost",
    "marginal_total_abatement_cost_(USD/tCO2e)"
]

# keep only relevant fields
df_merged_filtered = df_merged_filtered[relevant_fields]
df_merged_filtered.head()

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
1,Industrial carbon capture and sequestration,PFLO,130.422973,112.769234,-17.653739,3.076456,174.266545
4,Increase landfilling,WASO,130.422973,129.530159,-0.892814,0.016031,17.955288
5,Increase recycling,WASO,130.422973,126.709801,-3.713172,0.078710,21.197468
6,Improve manure management for poultry,LSMM,130.422973,129.946491,-0.476482,0.023736,49.814437
7,Improve rice management,AGRC,130.422973,130.027172,-0.395801,0.006703,16.936448


In [161]:
df_merged_filtered

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
1,Industrial carbon capture and sequestration,PFLO,130.422973,112.769234,-17.653739,3.076456,174.266545
4,Increase landfilling,WASO,130.422973,129.530159,-0.892814,0.016031,17.955288
5,Increase recycling,WASO,130.422973,126.709801,-3.713172,0.078710,21.197468
6,Improve manure management for poultry,LSMM,130.422973,129.946491,-0.476482,0.023736,49.814437
7,Improve rice management,AGRC,130.422973,130.027172,-0.395801,0.006703,16.936448
10,Consumer food waste reduction,WASO,130.422973,129.245310,-1.177663,-0.194755,-165.373757
15,Increase biogas capture at anaerobic decomposi...,LSMM,130.422973,130.415061,-0.007912,0.000396,49.999667
16,Incineration for energy production,WASO,130.422973,130.286710,-0.136263,0.012576,92.295177
18,Reduce use of HFCs,IPPU,130.422973,126.112759,-4.310214,0.064642,14.997514
19,Improve manure management for cattle and pigs,LSMM,130.422973,124.320026,-6.102947,0.170419,27.924088


In [162]:
df_merged_filtered.to_clipboard(index=False)

In [216]:
df_merged_filtered.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_for_QA.csv"), index=False)